# Dashboard Plotly - intoxicacao exogena em idosos

Este notebook replica os graficos centrais da analise inicial em Plotly e tambem prepara os dados que alimentam o dashboard local.

## Imports

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cwd = Path.cwd().resolve()
if (cwd / "utils").exists():
    ROOT = cwd
elif (cwd.parent / "utils").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Nao foi possivel localizar a raiz do projeto.")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

pd.options.display.float_format = "{:,.2f}".format

## Methods

In [ ]:
import importlib

from utils import intoxicacao_analysis as intox_analysis

intox_analysis = importlib.reload(intox_analysis)

DEFAULT_END_YEAR = intox_analysis.DEFAULT_END_YEAR
DEFAULT_SEX_PALETTE = intox_analysis.DEFAULT_SEX_PALETTE
DEFAULT_START_YEAR = intox_analysis.DEFAULT_START_YEAR
build_dashboard_payload = intox_analysis.build_dashboard_payload
build_descriptive_stats = intox_analysis.build_descriptive_stats
build_insight_lines = intox_analysis.build_insight_lines
build_overall_by_year = intox_analysis.build_overall_by_year
build_state_case_totals = intox_analysis.build_state_case_totals
build_state_summary = intox_analysis.build_state_summary
build_state_year = intox_analysis.build_state_year
build_top_groups_by_state = intox_analysis.build_top_groups_by_state
build_toxic_profile = intox_analysis.build_toxic_profile
build_year_totals = intox_analysis.build_year_totals
copy_plotly_bundle = intox_analysis.copy_plotly_bundle
filter_year_window = intox_analysis.filter_year_window
load_intoxicacao_tidy = intox_analysis.load_intoxicacao_tidy
summarize_year_coverage = intox_analysis.summarize_year_coverage
write_json_report = intox_analysis.write_json_report

## Params

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "Dados CFF - Intoxicação exógena .xlsx"
START_YEAR = DEFAULT_START_YEAR
END_YEAR = DEFAULT_END_YEAR
SEX_PALETTE = DEFAULT_SEX_PALETTE.copy()
DASHBOARD_DATA_PATH = ROOT / "dashboard" / "data" / "dashboard_data.json"
PLOTLY_BUNDLE_PATH = ROOT / "dashboard" / "assets" / "plotly.min.js"

params = pd.Series(
    {
        "data_path": str(DATA_PATH),
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "dashboard_data_path": str(DASHBOARD_DATA_PATH),
        "plotly_bundle_path": str(PLOTLY_BUNDLE_PATH),
    },
    name="value",
)
display(params.to_frame())

## Read

In [ ]:
df_raw = load_intoxicacao_tidy(DATA_PATH)
display(df_raw.head())
print(f"Linhas observadas na planilha: {len(df_raw):,}")

## Processing

In [ ]:
df = filter_year_window(df_raw, START_YEAR, END_YEAR)
coverage = summarize_year_coverage(df, START_YEAR, END_YEAR)
year_totals = build_year_totals(df, START_YEAR, END_YEAR)
state_summary = build_state_summary(year_totals)
descriptive_stats = build_descriptive_stats(year_totals)
overall_by_year = build_overall_by_year(year_totals)
state_case_totals = build_state_case_totals(year_totals)
state_year = build_state_year(year_totals)
toxic_profile = build_toxic_profile(df)
top_groups_by_state = build_top_groups_by_state(df, top_n=3)
insight_lines = build_insight_lines(
    year_totals=year_totals,
    state_summary=state_summary,
    toxic_profile=toxic_profile,
    coverage=coverage,
    start_year=START_YEAR,
    end_year=END_YEAR,
)
dashboard_payload = build_dashboard_payload(
    df=df,
    coverage=coverage,
    year_totals=year_totals,
    state_summary=state_summary,
    descriptive_stats=descriptive_stats,
    overall_by_year=overall_by_year,
    state_year=state_year,
    state_case_totals=state_case_totals,
    toxic_profile=toxic_profile,
    top_groups_by_state=top_groups_by_state,
    insight_lines=insight_lines,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

display(state_summary)
display(state_case_totals)
display(pd.DataFrame({"insights": insight_lines}))

## Plotly Graphs

In [ ]:
fig_overall = px.line(
    overall_by_year,
    x="year",
    y="total_year",
    color="sex_label",
    color_discrete_map=SEX_PALETTE,
    markers=True,
    title="Notificacoes anuais por sexo",
    labels={"year": "Ano", "total_year": "Notificacoes", "sex_label": "Sexo"},
)
fig_overall.update_layout(template="plotly_white", legend_title_text="Sexo")
fig_overall.show()

In [ ]:
heatmap_data = state_year.pivot(index="state", columns="year", values="total_year").fillna(0.0)
heatmap_norm = (heatmap_data.T / heatmap_data.max(axis=1)).T.fillna(0.0)

fig_heatmap = px.imshow(
    heatmap_norm,
    aspect="auto",
    color_continuous_scale=["#EDF4F7", "#BDD4DD", "#E5B8AE", "#C06A6A"],
    title="Intensidade anual por estado",
    labels={"x": "Ano", "y": "Estado", "color": "Indice relativo"},
)
fig_heatmap.update_layout(template="plotly_white")
fig_heatmap.show()

In [ ]:
for state in sorted(year_totals["state"].unique()):
    state_frame = year_totals.loc[year_totals["state"] == state].copy()
    fig_state = px.line(
        state_frame,
        x="year",
        y="total_year",
        color="sex_label",
        color_discrete_map=SEX_PALETTE,
        markers=True,
        title=f"{state} - trajetoria anual por sexo",
        labels={"year": "Ano", "total_year": "Notificacoes", "sex_label": "Sexo"},
    )
    fig_state.update_layout(template="plotly_white", height=440)
    fig_state.show()

In [ ]:
toxic_plot = toxic_profile.copy()
toxic_plot["share_pct"] = toxic_plot["share_within_sex"] * 100
top_toxic = toxic_plot.groupby("sex_label").head(8).copy()

fig_toxic = px.bar(
    top_toxic,
    x="count",
    y="toxic_group",
    color="sex_label",
    color_discrete_map=SEX_PALETTE,
    orientation="h",
    barmode="group",
    title="Principais grupos toxicos por sexo",
    labels={"count": "Notificacoes", "toxic_group": "Grupo toxico", "sex_label": "Sexo"},
)
fig_toxic.update_layout(template="plotly_white", yaxis={"categoryorder": "total ascending"})
fig_toxic.show()

In [ ]:
pie_total = state_case_totals.loc[state_case_totals["chart_group"] == "Total"].copy()
pie_f = state_case_totals.loc[state_case_totals["chart_group"] == "Feminino"].copy()
pie_m = state_case_totals.loc[state_case_totals["chart_group"] == "Masculino"].copy()

fig_pies = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "domain"}, {"type": "domain"}, {"type": "domain"}]],
    subplot_titles=["Total", "Feminino", "Masculino"],
)
for col_idx, frame in enumerate([pie_total, pie_f, pie_m], start=1):
    fig_pies.add_trace(
        go.Pie(labels=frame["state"], values=frame["total_year"], hole=0.2, sort=False),
        row=1,
        col=col_idx,
    )

fig_pies.update_layout(title_text="Distribuicao de casos por estado", template="plotly_white", height=520)
fig_pies.show()

## Write

In [ ]:
written_data_path = write_json_report(DASHBOARD_DATA_PATH, dashboard_payload)
written_plotly_bundle = copy_plotly_bundle(PLOTLY_BUNDLE_PATH)

print(f"Dashboard data: {written_data_path}")
print(f"Plotly bundle: {written_plotly_bundle}")
display(Markdown("### Principais achados\n" + "\n".join(f"- {line}" for line in insight_lines)))